In [1]:
RANDOM_STATE = 42
OUT_DIR = "runs"
RUN_NAME = "elliptic"

In [ ]:

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.base import clone
from scipy.stats import randint, uniform




DATA_DIR = r"D:\elliptic\Elliptic_Dataset"
TXS_FEATURES_FILE = "txs_features.csv"
TXS_CLASSES_FILE  = "txs_classes.csv"
TXS_EDGELIST_FILE = "txs_edgelist.csv" 

In [ ]:

import os

df_txs_features = pd.read_csv(os.path.join(DATA_DIR, TXS_FEATURES_FILE))
df_txs_classes  = pd.read_csv(os.path.join(DATA_DIR, TXS_CLASSES_FILE))

print("txs_features shape:", df_txs_features.shape)
print("txs_classes  shape:", df_txs_classes.shape)
print("txs_features columns:", df_txs_features.columns.tolist())
print("txs_classes  columns:", df_txs_classes.columns.tolist())


txs_features shape: (203769, 184)
txs_classes  shape: (203769, 2)
txs_features columns: ['txId', 'Time step', 'Local_feature_1', 'Local_feature_2', 'Local_feature_3', 'Local_feature_4', 'Local_feature_5', 'Local_feature_6', 'Local_feature_7', 'Local_feature_8', 'Local_feature_9', 'Local_feature_10', 'Local_feature_11', 'Local_feature_12', 'Local_feature_13', 'Local_feature_14', 'Local_feature_15', 'Local_feature_16', 'Local_feature_17', 'Local_feature_18', 'Local_feature_19', 'Local_feature_20', 'Local_feature_21', 'Local_feature_22', 'Local_feature_23', 'Local_feature_24', 'Local_feature_25', 'Local_feature_26', 'Local_feature_27', 'Local_feature_28', 'Local_feature_29', 'Local_feature_30', 'Local_feature_31', 'Local_feature_32', 'Local_feature_33', 'Local_feature_34', 'Local_feature_35', 'Local_feature_36', 'Local_feature_37', 'Local_feature_38', 'Local_feature_39', 'Local_feature_40', 'Local_feature_41', 'Local_feature_42', 'Local_feature_43', 'Local_feature_44', 'Local_feature_45',

In [ ]:

df = df_txs_features.merge(df_txs_classes, on="txId", how="left")

if "class" not in df.columns:
    raise ValueError("Không tìm thấy cột 'class' sau khi merge!")

print("\nPhân bố class ban đầu:")
print(df["class"].value_counts(dropna=False))


df = df.dropna(subset=["class"]).copy()
df["class"] = df["class"].astype(int)
df = df[df["class"] != 3].copy()

df["label"] = (df["class"] == 2).astype(int)

LABEL_COL = "label"
print("\nPhân bố label sau khi bỏ unknown (0=licit,1=illicit):")
print(df[LABEL_COL].value_counts())


Phân bố class ban đầu:
class
3    157205
2     42019
1      4545
Name: count, dtype: int64

Phân bố label sau khi bỏ unknown (0=licit,1=illicit):
label
1    42019
0     4545
Name: count, dtype: int64


In [ ]:

possible_ts_cols = ["Time step", "time_step"]
ts_col = None
for c in possible_ts_cols:
    if c in df.columns:
        ts_col = c
        break

if ts_col is None:
    raise ValueError("Không tìm thấy cột time-step (ví dụ 'Time step' hoặc 'time_step')!")

drop_id_cols = ["txId", ts_col]
drop_label_cols = ["class", LABEL_COL]

cols_to_drop = [c for c in drop_id_cols + drop_label_cols if c in df.columns]

feature_cols = [c for c in df.columns if c not in cols_to_drop]

X_df = df[feature_cols].copy()
y = df[LABEL_COL].values
time_steps = df[ts_col].values

print("\nSố feature:", len(feature_cols))
print("Một vài feature đầu:", feature_cols[:10])

num_cols = X_df.select_dtypes(include=[np.number]).columns.tolist()
X_df = X_df[num_cols].copy()
print("\nSố feature numeric:", len(num_cols))


Số feature: 182
Một vài feature đầu: ['Local_feature_1', 'Local_feature_2', 'Local_feature_3', 'Local_feature_4', 'Local_feature_5', 'Local_feature_6', 'Local_feature_7', 'Local_feature_8', 'Local_feature_9', 'Local_feature_10']

Số feature numeric: 182


In [ ]:

from sklearn.model_selection import train_test_split


X_all = X_df.values
y_all = y

X_train, X_tmp, y_train, y_tmp = train_test_split(
    X_all, y_all,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y_all
)

X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_tmp
)

print("\nKích thước:")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val  :", X_val.shape,   "y_val  :", y_val.shape)
print("X_test :", X_test.shape,  "y_test :", y_test.shape)

print("\nPhân bố nhãn train:")
print(pd.Series(y_train).value_counts())
print("\nPhân bố nhãn val:")
print(pd.Series(y_val).value_counts())
print("\nPhân bố nhãn test:")
print(pd.Series(y_test).value_counts())



Kích thước:
X_train: (32594, 182) y_train: (32594,)
X_val  : (6985, 182) y_val  : (6985,)
X_test : (6985, 182) y_test : (6985,)

Phân bố nhãn train:
1    29413
0     3181
Name: count, dtype: int64

Phân bố nhãn val:
1    6303
0     682
Name: count, dtype: int64

Phân bố nhãn test:
1    6303
0     682
Name: count, dtype: int64


In [ ]:

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)


In [ ]:

def sample_param(dist, rng):
    """Lấy 1 giá trị từ distribution hoặc list."""
    if hasattr(dist, "rvs"):
        
        return dist.rvs(random_state=rng)
    
    dist = list(dist)
    return dist[rng.randint(0, len(dist))]

def random_search_single_model(
    name,
    base_estimator,
    param_dist,
    X_train, y_train,
    X_val,   y_val,
    n_iter=30,
    scoring="macro"
):

    print(f"\n===== Random search cho {name} (không k-fold, dùng VAL) =====")
    rng = np.random.RandomState(RANDOM_STATE)
    best_f1 = -1.0
    best_params = None

    for i in range(n_iter):
        params = {k: sample_param(v, rng) for k, v in param_dist.items()}

        model = clone(base_estimator)
        model.set_params(**params)
        model.fit(X_train, y_train)

        y_val_pred = model.predict(X_val)
        # macro-F1 dùng để chọn model
        f1_macro = f1_score(y_val, y_val_pred, average="macro")

        print(f"Iter {i+1:02d}/{n_iter}: F1_macro(val) = {f1_macro:.4f}, params = {params}")

        if f1_macro > best_f1:
            best_f1 = f1_macro
            best_params = params

    print(f"\n>>> {name} – best F1_macro(val) = {best_f1:.4f}")
    print("Best params:", best_params)


    X_train_full = np.vstack([X_train, X_val])
    y_train_full = np.concatenate([y_train, y_val])

    best_model = clone(base_estimator)
    best_model.set_params(**best_params)
    best_model.fit(X_train_full, y_train_full)

    return best_model


n_pos = np.sum(y_train == 1)
n_neg = np.sum(y_train == 0)
scale_pos_weight = n_neg / n_pos if n_pos > 0 else 1.0
print("\nscale_pos_weight (train):", scale_pos_weight)


scale_pos_weight (train): 0.10814945772277565


In [ ]:

# 1) RandomForest
rf_base = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
rf_param_dist = {
    "n_estimators": randint(200, 600),
    "max_depth": randint(3, 30),
    "min_samples_split": randint(2, 50),
    "min_samples_leaf": randint(1, 20),
    "max_features": ["sqrt", "log2", None],
    "class_weight": [None, "balanced"],
    "criterion": ["gini", "entropy"], 
}

best_rf = random_search_single_model(
    "RandomForest",
    rf_base,
    rf_param_dist,
    X_train_scaled, y_train,
    X_val_scaled,   y_val,
    n_iter=30
)







===== Random search cho RandomForest (không k-fold, dùng VAL) =====
Iter 01/30: F1_macro(val) = 0.9514, params = {'n_estimators': 302, 'max_depth': 22, 'min_samples_split': 30, 'min_samples_leaf': 15, 'max_features': None, 'class_weight': 'balanced', 'criterion': 'gini'}
Iter 02/30: F1_macro(val) = 0.9103, params = {'n_estimators': 220, 'max_depth': 9, 'min_samples_split': 20, 'min_samples_leaf': 11, 'max_features': None, 'class_weight': 'balanced', 'criterion': 'gini'}
Iter 03/30: F1_macro(val) = 0.9354, params = {'n_estimators': 299, 'max_depth': 10, 'min_samples_split': 25, 'min_samples_leaf': 3, 'max_features': 'log2', 'class_weight': None, 'criterion': 'entropy'}
Iter 04/30: F1_macro(val) = 0.9563, params = {'n_estimators': 543, 'max_depth': 14, 'min_samples_split': 31, 'min_samples_leaf': 6, 'max_features': 'log2', 'class_weight': 'balanced', 'criterion': 'entropy'}
Iter 05/30: F1_macro(val) = 0.8678, params = {'n_estimators': 476, 'max_depth': 3, 'min_samples_split': 13, 'min_s

In [ ]:

xgb_base = XGBClassifier(
    random_state=RANDOM_STATE,
    tree_method="hist",   
    eval_metric="logloss",
    use_label_encoder=False
)

xgb_param_dist = {
    "n_estimators": randint(200, 800),
    "max_depth": randint(3, 12),
    "learning_rate": uniform(0.01, 0.29),
    "subsample": uniform(0.6, 0.4),     
    "colsample_bytree": uniform(0.6, 0.4), 
    "min_child_weight": randint(1, 10),
    "reg_lambda": uniform(0, 5),
    "objective": ["binary:logistic"],     
    "eval_metric": ["logloss", "aucpr"],   
    "scale_pos_weight": [scale_pos_weight]
}

best_xgb = random_search_single_model(
    "XGBoost",
    xgb_base,
    xgb_param_dist,
    X_train_scaled, y_train,
    X_val_scaled,   y_val,
    n_iter=30
)


===== Random search cho XGBoost (không k-fold, dùng VAL) =====


d:\elliptic\venv1\Lib\site-packages\xgboost\core.py:158: UserWarning: [00:26:39] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Iter 01/30: F1_macro(val) = 0.9695, params = {'n_estimators': 302, 'max_depth': 6, 'learning_rate': np.float64(0.28570714885887566), 'subsample': np.float64(0.892797576724562), 'colsample_bytree': np.float64(0.8394633936788146), 'min_child_weight': 7, 'reg_lambda': np.float64(2.229163764267956), 'objective': 'binary:logistic', 'eval_metric': 'logloss', 'scale_pos_weight': np.float64(0.10814945772277565)}


d:\elliptic\venv1\Lib\site-packages\xgboost\core.py:158: UserWarning: [00:26:41] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Iter 02/30: F1_macro(val) = 0.9646, params = {'n_estimators': 530, 'max_depth': 10, 'learning_rate': np.float64(0.10677549723031632), 'subsample': np.float64(0.6571467271687763), 'colsample_bytree': np.float64(0.8603553891795411), 'min_child_weight': 5, 'reg_lambda': np.float64(4.8495492608099715), 'objective': 'binary:logistic', 'eval_metric': 'aucpr', 'scale_pos_weight': np.float64(0.10814945772277565)}


...

d:\elliptic\venv1\Lib\site-packages\xgboost\core.py:158: UserWarning: [00:28:01] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Iter 29/30: F1_macro(val) = 0.9732, params = {'n_estimators': 330, 'max_depth': 6, 'learning_rate': np.float64(0.267255063036884), 'subsample': np.float64(0.9119502183430496), 'colsample_bytree': np.float64(0.8568126584617151), 'min_child_weight': 1, 'reg_lambda': np.float64(0.8081435704730688), 'objective': 'binary:logistic', 'eval_metric': 'aucpr', 'scale_pos_weight': np.float64(0.10814945772277565)}


d:\elliptic\venv1\Lib\site-packages\xgboost\core.py:158: UserWarning: [00:28:04] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Iter 30/30: F1_macro(val) = 0.9638, params = {'n_estimators': 303, 'max_depth': 6, 'learning_rate': np.float64(0.17839912021657184), 'subsample': np.float64(0.7489131066246972), 'colsample_bytree': np.float64(0.9760533769831113), 'min_child_weight': 9, 'reg_lambda': np.float64(0.025307919231093434), 'objective': 'binary:logistic', 'eval_metric': 'aucpr', 'scale_pos_weight': np.float64(0.10814945772277565)}

>>> XGBoost – best F1_macro(val) = 0.9749
Best params: {'n_estimators': 329, 'max_depth': 7, 'learning_rate': np.float64(0.15194130048049329), 'subsample': np.float64(0.9942601816442402), 'colsample_bytree': np.float64(0.6968221086046001), 'min_child_weight': 4, 'reg_lambda': np.float64(0.40426663166357624), 'objective': 'binary:logistic', 'eval_metric': 'aucpr', 'scale_pos_weight': np.float64(0.10814945772277565)}


d:\elliptic\venv1\Lib\site-packages\xgboost\core.py:158: UserWarning: [00:28:06] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


In [ ]:
# 3) LightGBM
lgbm_base = LGBMClassifier(
    random_state=RANDOM_STATE,
    objective="binary",
    n_jobs=-1
)

lgbm_param_dist = {
    "n_estimators": randint(200, 800),
    "num_leaves": randint(15, 255),
    "max_depth": randint(3, 12),        
    "learning_rate": uniform(0.01, 0.29),
    "subsample": uniform(0.6, 0.4),         
    "colsample_bytree": uniform(0.6, 0.4),  
    "min_child_samples": randint(10, 100),
    "reg_lambda": uniform(0, 5),
    "objective": ["binary"],             
    "metric": ["binary_logloss", "auc", "aucpr"],
    "scale_pos_weight": [scale_pos_weight]
} 

best_lgbm = random_search_single_model(
    "LightGBM",
    lgbm_base,
    lgbm_param_dist,
    X_train_scaled, y_train,
    X_val_scaled,   y_val,
    n_iter=30
)


===== Random search cho LightGBM (không k-fold, dùng VAL) =====
[LightGBM] [Info] Number of positive: 29413, number of negative: 3181
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.013544 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 42090
[LightGBM] [Info] Number of data points in the train set: 32594, number of used features: 181
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.902405 -> initscore=2.224241
[LightGBM] [Info] Start training from score 2.224241
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 29413, number of negative: 3181
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.013751 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 42090
[LightGBM] [Info] Number of data points in the train set: 32594, number of used features: 181
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.902405 -> initscore=2.224241
[LightGBM] [Info] Start training from score 2.224241
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain

d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 29413, number of negative: 3181
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.013032 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 42090
[LightGBM] [Info] Number of data points in the train set: 32594, number of used features: 181
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.902405 -> initscore=2.224241
[LightGBM] [Info] Start training from score 2.224241
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain

...

Iter 28/30: F1_macro(val) = 0.9758, params = {'n_estimators': 703, 'num_leaves': 142, 'max_depth': 10, 'learning_rate': np.float64(0.2562747890433116), 'subsample': np.float64(0.8630451569201374), 'colsample_bytree': np.float64(0.8273234413341887), 'min_child_samples': 85, 'reg_lambda': np.float64(2.5440703841938), 'objective': 'binary', 'metric': 'auc', 'scale_pos_weight': np.float64(0.10814945772277565)}
[LightGBM] [Info] Number of positive: 29413, number of negative: 3181
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.022033 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 42090
[LightGBM] [Info] Number of data points in the train set: 32594, number of used features: 181
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.902405 -> initscore=2.224241
[LightGBM] [Info] Start training from score 2.224241
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] N

d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Iter 29/30: F1_macro(val) = 0.9743, params = {'n_estimators': 574, 'num_leaves': 36, 'max_depth': 8, 'learning_rate': np.float64(0.2938789288997526), 'subsample': np.float64(0.794696861183782), 'colsample_bytree': np.float64(0.9624395150874216), 'min_child_samples': 36, 'reg_lambda': np.float64(3.974056517708242), 'objective': 'binary', 'metric': 'auc', 'scale_pos_weight': np.float64(0.10814945772277565)}
[LightGBM] [Info] Number of positive: 29413, number of negative: 3181
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.023290 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 42090
[LightGBM] [Info] Number of data points in the train set: 32594, number of used features: 181
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.902405 -> initscore=2.224241
[LightGBM] [Info] Start training from score 2.224241
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No

d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 35716, number of negative: 3863
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.017440 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 42227
[LightGBM] [Info] Number of data points in the train set: 39579, number of used features: 181
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.902398 -> initscore=2.224155
[LightGBM] [Info] Start training from score 2.224155
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain

In [ ]:

from itertools import combinations
from sklearn.metrics import roc_auc_score

base_models = {
    "RF": best_rf,
    "XGB": best_xgb,
    "LGBM": best_lgbm
}

def _proba_pos(model, X):
    proba = model.predict_proba(X)
    if proba.ndim == 2 and proba.shape[1] >= 2:
        return proba[:, 1]
    return proba.ravel()

def ensemble_predict(models_dict, X, voting="soft", threshold=0.5):
    models = list(models_dict.values())

    probas = np.vstack([_proba_pos(m, X) for m in models])  
    proba_mean = probas.mean(axis=0)

    if voting == "soft":
        y_pred = (proba_mean >= threshold).astype(int)
        return y_pred, proba_mean

    if voting == "hard":
        preds = np.vstack([m.predict(X) for m in models]).astype(int)
        votes = preds.sum(axis=0)
        half = len(models) / 2

        y_pred = (votes > half).astype(int)

        tie_mask = (votes == half)
        if np.any(tie_mask):
            y_pred[tie_mask] = (proba_mean[tie_mask] >= threshold).astype(int)

        return y_pred, proba_mean

    raise ValueError("voting phải là 'hard' hoặc 'soft'")

def eval_binary(y_true, y_pred, y_score=None):
    out = {}
    out["accuracy"] = accuracy_score(y_true, y_pred)
    out["f1_binary"] = f1_score(y_true, y_pred)
    out["f1_micro"]  = f1_score(y_true, y_pred, average="micro")
    out["f1_macro"]  = f1_score(y_true, y_pred, average="macro")
    if y_score is not None:
        try:
            out["roc_auc"] = roc_auc_score(y_true, y_score)
        except Exception:
            out["roc_auc"] = np.nan
    return out

results = []


combos = list(combinations(base_models.keys(), 2)) + [tuple(base_models.keys())]

for combo in combos:
    combo_models = {k: base_models[k] for k in combo}
    combo_name = "+".join(combo)

    for voting in ["hard", "soft"]:
        y_pred, y_score = ensemble_predict(combo_models, X_test_scaled, voting=voting, threshold=0.5)
        m = eval_binary(y_test, y_pred, y_score)
        
        results.append({"ensemble": combo_name, "voting": voting, **m})

        print(f"\n===== Ensemble [{combo_name}] | {voting.upper()} vote =====")
        print(classification_report(y_test, y_pred, digits=4))
        print("Accuracy:", m["accuracy"])
        print("F1 (binary, pos_label=1):", m["f1_binary"])
        print("F1 micro:", m["f1_micro"])
        print("F1 macro:", m["f1_macro"])
        print("ROC-AUC:", m.get("roc_auc", None))
        print("Confusion matrix:")
        print(confusion_matrix(y_test, y_pred))

results_df = pd.DataFrame(results).sort_values(
    by=["f1_binary", "roc_auc", "accuracy"],
    ascending=False
)
display(results_df)



===== Ensemble [RF+XGB] | HARD vote =====
              precision    recall  f1-score   support

           0     0.9672    0.9516    0.9593       682
           1     0.9948    0.9965    0.9956      6303

    accuracy                         0.9921      6985
   macro avg     0.9810    0.9741    0.9775      6985
weighted avg     0.9921    0.9921    0.9921      6985

Accuracy: 0.9921259842519685
F1 (binary, pos_label=1): 0.995640802092415
F1 micro: 0.9921259842519685
F1 macro: 0.977495197794175
ROC-AUC: 0.9977141639483686
Confusion matrix:
[[ 649   33]
 [  22 6281]]

===== Ensemble [RF+XGB] | SOFT vote =====
              precision    recall  f1-score   support

           0     0.9672    0.9516    0.9593       682
           1     0.9948    0.9965    0.9956      6303

    accuracy                         0.9921      6985
   macro avg     0.9810    0.9741    0.9775      6985
weighted avg     0.9921    0.9921    0.9921      6985

Accuracy: 0.9921259842519685
F1 (binary, pos_label=1): 0.

d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



===== Ensemble [RF+LGBM] | HARD vote =====
              precision    recall  f1-score   support

           0     0.9877    0.9457    0.9663       682
           1     0.9942    0.9987    0.9964      6303

    accuracy                         0.9936      6985
   macro avg     0.9910    0.9722    0.9814      6985
weighted avg     0.9935    0.9936    0.9935      6985

Accuracy: 0.9935576234788833
F1 (binary, pos_label=1): 0.996438464582509
F1 micro: 0.9935576234788833
F1 macro: 0.9813652997069848
ROC-AUC: 0.9977597597010779
Confusion matrix:
[[ 645   37]
 [   8 6295]]

===== Ensemble [RF+LGBM] | SOFT vote =====
              precision    recall  f1-score   support

           0     0.9877    0.9457    0.9663       682
           1     0.9942    0.9987    0.9964      6303

    accuracy                         0.9936      6985
   macro avg     0.9910    0.9722    0.9814      6985
weighted avg     0.9935    0.9936    0.9935      6985

Accuracy: 0.9935576234788833
F1 (binary, pos_label=1):

d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



===== Ensemble [XGB+LGBM] | HARD vote =====
              precision    recall  f1-score   support

           0     0.9803    0.9487    0.9642       682
           1     0.9945    0.9979    0.9962      6303

    accuracy                         0.9931      6985
   macro avg     0.9874    0.9733    0.9802      6985
weighted avg     0.9931    0.9931    0.9931      6985

Accuracy: 0.9931281317108088
F1 (binary, pos_label=1): 0.9961989230281914
F1 micro: 0.9931281317108088
F1 macro: 0.9802157059254221
ROC-AUC: 0.9970513971143471
Confusion matrix:
[[ 647   35]
 [  13 6290]]

===== Ensemble [XGB+LGBM] | SOFT vote =====
              precision    recall  f1-score   support

           0     0.9803    0.9487    0.9642       682
           1     0.9945    0.9979    0.9962      6303

    accuracy                         0.9931      6985
   macro avg     0.9874    0.9733    0.9802      6985
weighted avg     0.9931    0.9931    0.9931      6985

Accuracy: 0.9931281317108088
F1 (binary, pos_label=

d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



===== Ensemble [RF+XGB+LGBM] | HARD vote =====
              precision    recall  f1-score   support

           0     0.9802    0.9457    0.9627       682
           1     0.9942    0.9979    0.9960      6303

    accuracy                         0.9928      6985
   macro avg     0.9872    0.9718    0.9794      6985
weighted avg     0.9928    0.9928    0.9928      6985

Accuracy: 0.9928418038654259
F1 (binary, pos_label=1): 0.9960411718131433
F1 micro: 0.9928418038654259
F1 macro: 0.9793638694886613
ROC-AUC: 0.997785349154129
Confusion matrix:
[[ 645   37]
 [  13 6290]]

===== Ensemble [RF+XGB+LGBM] | SOFT vote =====
              precision    recall  f1-score   support

           0     0.9773    0.9487    0.9628       682
           1     0.9945    0.9976    0.9960      6303

    accuracy                         0.9928      6985
   macro avg     0.9859    0.9732    0.9794      6985
weighted avg     0.9928    0.9928    0.9928      6985

Accuracy: 0.9928418038654259
F1 (binary, pos_l

d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,ensemble,voting,accuracy,f1_binary,f1_micro,f1_macro,roc_auc
2,RF+LGBM,hard,0.993558,0.996438,0.993558,0.981365,0.997760
3,RF+LGBM,soft,0.993558,0.996438,0.993558,0.981365,0.997760
4,XGB+LGBM,hard,0.993128,0.996199,0.993128,0.980216,0.997051
5,XGB+LGBM,soft,0.993128,0.996199,0.993128,0.980216,0.997051
6,RF+XGB+LGBM,hard,0.992842,0.996041,0.992842,0.979364,0.997785
7,RF+XGB+LGBM,soft,0.992842,0.996040,0.992842,0.979419,0.997785
0,RF+XGB,hard,0.992126,0.995641,0.992126,0.977495,0.997714
1,RF+XGB,soft,0.992126,0.995641,0.992126,0.977495,0.997714


In [ ]:
from sklearn.metrics import roc_auc_score


def evaluate_on_test(name, model, X_test, y_test):
    print(f"\n===== {name} trên TEST =====")
    y_pred = model.predict(X_test)

    acc      = accuracy_score(y_test, y_pred)
    f1_bin   = f1_score(y_test, y_pred)       
    f1_micro = f1_score(y_test, y_pred, average="micro")
    f1_macro = f1_score(y_test, y_pred, average="macro")


    auc_roc = None
    try:
        if hasattr(model, "predict_proba"):

            y_score = model.predict_proba(X_test)[:, 1]
            auc_roc = roc_auc_score(y_test, y_score)
        elif hasattr(model, "decision_function"):

            y_score = model.decision_function(X_test)
            auc_roc = roc_auc_score(y_test, y_score)
    except Exception as e:
        print(f"[WARN] Không tính được AUC-ROC: {e}")

    print("Accuracy :", acc)
    print("F1 (binary, pos_label=1):", f1_bin)
    print("F1 micro :", f1_micro)
    print("F1 macro :", f1_macro)
    if auc_roc is not None:
        print("AUC-ROC :", auc_roc)

    print("\nclassification_report:")
    print(classification_report(y_test, y_pred, digits=4))

    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_pred))


models = {
    "RandomForest": best_rf,
    "XGBoost":      best_xgb,
    "LightGBM":     best_lgbm,
}

for name, model in models.items():
    evaluate_on_test(name, model, X_test_scaled, y_test)


===== RandomForest trên TEST =====
Accuracy : 0.9894058697208303
F1 (binary, pos_label=1): 0.9941427892987178
F1 micro : 0.9894058697208303
F1 macro : 0.9693767838709157
AUC-ROC : 0.997159803342727

classification_report:
              precision    recall  f1-score   support

           0     0.9648    0.9252    0.9446       682
           1     0.9919    0.9964    0.9941      6303

    accuracy                         0.9894      6985
   macro avg     0.9784    0.9608    0.9694      6985
weighted avg     0.9893    0.9894    0.9893      6985

Confusion matrix:
[[ 631   51]
 [  23 6280]]

===== XGBoost trên TEST =====
Accuracy : 0.9922691481746599
F1 (binary, pos_label=1): 0.9957176843774782
F1 micro : 0.9922691481746599
F1 macro : 0.9780059010122686
AUC-ROC : 0.9969411298348363

classification_report:
              precision    recall  f1-score   support

           0     0.9631    0.9575    0.9603       682
           1     0.9954    0.9960    0.9957      6303

    accuracy          

d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
# =========================================
# 10. SHAP cho Elliptic++ (3 model RF, XGB, LGBM)
# Vẽ kiểu Springer: (a) Heatmap + (b) Grouped bar
# =========================================
try:
    import shap
    shap_available = True
except Exception as e:
    print("Không thể import SHAP (shap). Hãy cài: pip install shap")
    print(e)
    shap_available = False

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

if shap_available and X_test_scaled.shape[0] > 0:
    shap.initjs()

    # -------- 1. STYLE SPRINGER (y như code mẫu) --------
    plt.rcParams['font.family'] = 'serif'
    plt.rcParams['font.serif'] = ['Times New Roman', 'DejaVu Serif', 'Liberation Serif']
    plt.rcParams['font.size'] = 12
    plt.rcParams['axes.labelsize'] = 14
    plt.rcParams['axes.titlesize'] = 16
    plt.rcParams['xtick.labelsize'] = 11
    plt.rcParams['ytick.labelsize'] = 11
    plt.rcParams['legend.fontsize'] = 11
    plt.rcParams['figure.titlesize'] = 18

    # -------- 2. TÍNH SHAP: Mean |SHAP| cho Elliptic++ --------
    n_background = min(200,X_train_scaled.shape[0])
    n_explain    = min(200,X_test_scaled.shape[0])

    X_background = X_train_scaled[:n_background]
    X_explain    = X_test_scaled[:n_explain]

    print(f"[Elliptic] Background SHAP size: {X_background.shape}")
    print(f"[Elliptic] Explain SHAP size   : {X_explain.shape}")

    models_to_explain = {
        "RandomForest": best_rf,
        "XGBoost":      best_xgb,
        "LightGBM":     best_lgbm,
    }

    mean_abs_dict = {}   # model_name -> (n_features,)

    for name, model in models_to_explain.items():
        print(f"\n================ SHAP cho {name} (Elliptic++) ================")

        # TreeExplainer, model_output="probability" cho cùng thang
        explainer = shap.TreeExplainer(
            model,
            data=X_background,
            feature_perturbation="interventional",
            model_output="probability"
        )

        sv = explainer.shap_values(
            X_explain,
            check_additivity=False
        )

        # ----- Chuẩn hoá về SHAP cho class=1, sv_pos: (n_samples, n_features) -----
        if isinstance(sv, list):
            if hasattr(model, "classes_"):
                cls_idx = int(np.where(model.classes_ == 1)[0][0])
            else:
                cls_idx = 1
            sv_pos = np.array(sv[cls_idx])

        elif isinstance(sv, np.ndarray) and sv.ndim == 3:
            if hasattr(model, "classes_"):
                cls_idx = int(np.where(model.classes_ == 1)[0][0])
            else:
                cls_idx = sv.shape[2] - 1
            sv_pos = np.array(sv[:, :, cls_idx])

        else:
            sv_pos = np.array(sv)

        assert sv_pos.ndim == 2, f"sv_pos không phải 2D cho model {name} (shape={sv_pos.shape})"
        print("  sv_pos.shape =", sv_pos.shape, "| ndim =", sv_pos.ndim)

        # Mean absolute SHAP value cho mỗi feature
        mean_abs = np.mean(np.abs(sv_pos), axis=0)   # shape = (n_features,)
        mean_abs_dict[name] = mean_abs

    # -------- 3. TẠO BẢNG MEAN |SHAP| (giống "Importance" code mẫu) --------
    df_mean = pd.DataFrame(mean_abs_dict, index=feature_cols)

    # chọn top_n feature theo trung bình across models
    top_n = 10 if len(feature_cols) >= 10 else len(feature_cols)
    df_mean['avg'] = df_mean.mean(axis=1)
    df_mean = df_mean.sort_values('avg', ascending=False)
    df_top = df_mean.head(top_n).drop(columns='avg')   # (top_n x 3)

    # -------- 4. VẼ (a) Heatmap + (b) Grouped bar – SPRINGER --------
    plt.close('all')
    fig, axes = plt.subplots(
        1, 2,
        figsize=(11, 4.3),   # 1 dòng trong layout (11, 13) của code mẫu
        dpi=600,
        constrained_layout=True
    )
    ax0, ax1 = axes

    # ---------- (a) Heatmap: KHÔNG CHUẨN HOÁ, dùng Mean |SHAP| gốc ----------
    sns.heatmap(
        df_top,
        annot=True,
        fmt=".4f",                 # SHAP thật thường nhỏ, 4 số lẻ
        cmap="YlOrRd",
        linewidths=0.5,
        linecolor='gray',
        cbar_kws={"label": "Mean", "shrink": 0.8},
        ax=ax0,
        annot_kws={"size": 11}
    )

    ax0.set_title("(a) Elliptic++ Heatmap", pad=10, fontweight='bold')
    ax0.set_xlabel("")
    ax0.set_ylabel("")
    ax0.set_yticklabels(df_top.index, rotation=0)
    ax0.set_xticklabels(df_top.columns, rotation=0)

    # ---------- (b) Grouped bar: giống plot_grouped_bar code mẫu ----------
    df_long = (
        df_top
        .reset_index()
        .melt(id_vars="index", var_name="Model", value_name="Importance")
        .rename(columns={"index": "Feature"})
    )

    feature_order = df_top.index.tolist()

    sns.barplot(
        data=df_long,
        x="Importance",
        y="Feature",
        hue="Model",
        order=feature_order,
        palette=["#1f77b4", "#d62728", "#2ca02c"],  # giống code mẫu
        ax=ax1,
        edgecolor='black',
        linewidth=0.8
    )

    ax1.set_title("(b) Elliptic++", pad=10, fontweight='bold')
    ax1.set_xlabel("Mean")
    ax1.set_ylabel("")
    ax1.grid(axis="x", linestyle='--', alpha=0.5)

    ax1.legend(
        title="Model",
        frameon=True,
        fancybox=False,
        edgecolor='black'
    )

    plt.show()

    # (tuỳ chọn) nếu vẫn muốn giải thích local cho 1 node bằng XGBoost thì có thể thêm block riêng ở dưới,
    # tương tự như bạn làm cho Ethereum/BLTE.
else:
    print("Bỏ qua SHAP Elliptic++ vì không sẵn sàng hoặc không có mẫu test.")


In [ ]:
    # -------- 3. TẠO BẢNG MEAN |SHAP| (giống "Importance" code mẫu) --------
    df_mean = pd.DataFrame(mean_abs_dict, index=feature_cols)

    # chọn top_n feature theo trung bình across models
    top_n = 6 if len(feature_cols) >= 10 else len(feature_cols)
    df_mean['avg'] = df_mean.mean(axis=1)
    df_mean = df_mean.sort_values('avg', ascending=False)
    df_top = df_mean.head(top_n).drop(columns='avg')   # (top_n x 3)

    # -------- 4. VẼ (a) Heatmap + (b) Grouped bar – SPRINGER --------
    plt.close('all')
    fig, axes = plt.subplots(
        1, 2,
        figsize=(11, 4.3),   # 1 dòng trong layout (11, 13) của code mẫu
        dpi=600,
        constrained_layout=True
    )
    ax0, ax1 = axes

    # ---------- (a) Heatmap: KHÔNG CHUẨN HOÁ, dùng Mean |SHAP| gốc ----------
    sns.heatmap(
        df_top,
        annot=True,
        fmt=".4f",                 # SHAP thật thường nhỏ, 4 số lẻ
        cmap="YlOrRd",
        linewidths=0.5,
        linecolor='gray',
        cbar_kws={"label": "Mean", "shrink": 0.8},
        ax=ax0,
        annot_kws={"size": 11}
    )

    ax0.set_title("(a) Elliptic++", pad=10, fontweight='bold')
    ax0.set_xlabel("")
    ax0.set_ylabel("")
    ax0.set_yticklabels(df_top.index, rotation=0)
    ax0.set_xticklabels(df_top.columns, rotation=0)

    # ---------- (b) Grouped bar: giống plot_grouped_bar code mẫu ----------
    df_long = (
        df_top
        .reset_index()
        .melt(id_vars="index", var_name="Model", value_name="Importance")
        .rename(columns={"index": "Feature"})
    )

    feature_order = df_top.index.tolist()

    sns.barplot(
        data=df_long,
        x="Importance",
        y="Feature",
        hue="Model",
        order=feature_order,
        palette=["#1f77b4", "#d62728", "#2ca02c"],  # giống code mẫu
        ax=ax1,
        edgecolor='black',
        linewidth=0.8
    )

    ax1.set_title("(b) Elliptic++", pad=10, fontweight='bold')
    ax1.set_xlabel("Mean")
    ax1.set_ylabel("")
    ax1.grid(axis="x", linestyle='--', alpha=0.5)

    ax1.legend(
        title="Model",
        frameon=True,
        fancybox=False,
        edgecolor='black'
    )

    fig.savefig("eliptic.png", dpi=600, bbox_inches="tight")

    # (tuỳ chọn) nếu vẫn muốn giải thích local cho 1 node bằng XGBoost thì có thể thêm block riêng ở dưới,
    # tương tự như bạn làm cho Ethereum/BLTE.


In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.metrics import f1_score

rows = []

# ===== 3 base models =====
base_models = {"RF": best_rf, "XGB": best_xgb, "LGBM": best_lgbm}
for name, model in base_models.items():
    y_pred = model.predict(X_test_scaled)
    rows.append({
        "seed": int(RANDOM_STATE),
        "model": name,
        "f1_macro": float(f1_score(y_test, y_pred, average="macro"))
    })

# ===== 8 ensemble models =====
if "results_df" in globals():
    tmp = results_df.copy()
    tmp["model"] = tmp["ensemble"].astype(str) + "_" + tmp["voting"].astype(str)
    for _, r in tmp.iterrows():
        rows.append({
            "seed": int(RANDOM_STATE),
            "model": str(r["model"]),
            "f1_macro": float(r["f1_macro"])
        })
else:
    print("WARNING: results_df chưa tồn tại -> chỉ lưu macro-F1 cho 3 base models.")

# ===== LƯU CSV (append) =====
out_csv = Path(r"D:\elliptic\ellipticv2\runs\macro_f1_all_models_by_seed.csv")
out_csv.parent.mkdir(parents=True, exist_ok=True)

df_out = pd.DataFrame(rows)

write_header = (not out_csv.exists()) or out_csv.stat().st_size == 0
df_out.to_csv(out_csv, mode="a", header=write_header, index=False)

print("Saved:", out_csv, "| rows:", len(df_out))
